# 第10課 - AI 代理於生產環境

在本課程中，你將學習如何使用 Microsoft Agent Framework (Python) 為 AI 代理設計 **生產模式**。我們會涵蓋：

- **可觀察性** — 在代理互動中加入計時與日誌記錄
- **評估** — 使用評估代理為回應質量打分
- **成本管理** — 代幣優化與模型選擇策略

情境是一個 **旅行代理**，幫助用戶規劃行程，並在其上加入監控與評估層。


## 設定


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## 生產考量

將 AI 代理從原型推向生產環境需要細心留意三大支柱：

1. **可觀察性** — 你需要能夠看見代理在做什麼、花了多久，以及呼叫了哪些工具。沒有追蹤與日誌記錄，要在生產環境中排查問題幾乎不可能。

2. **評估** — 自動化的品質檢查可確保代理的回應隨時間保持準確、完整且有幫助。評估代理可以根據既定標準為回應打分。

3. **成本管理** — 詞元使用量會直接影響成本。像是提示優化、模型選擇及快取等策略可以在不犧牲品質的情況下，幫助控制開支。


## 建立一個可觀察的代理

我們定義旅遊工具，並以計時封裝代理呼叫，以便監測延遲。在生產環境中，你會將其與 OpenTelemetry 或類似的追蹤後端整合。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 評估模式

在生產環境中，一個常見的模式是使用第二個代理作為 **評估者**。評估者會根據預先定義的準則，例如完整性、正確性和有用性，對主要代理的回應進行評分。

這使得：
- 在回應到達使用者之前的自動化品質把關
- 當提示或模型更改時的回歸檢測
- 持續監控代理的表現隨時間變化


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 成本管理策略

控制成本對生產環境的 AI agent 至關重要。以下是主要策略：

| 策略 | 說明 |
|---|---|
| **提示優化** | 系統指示要保持簡潔。移除多餘的上下文以減少輸入 tokens。 |
| **模型選擇** | 對於像分類或抽取之類的簡單任務，使用較小且成本較低的模型（例如 GPT-4o-mini），並把較大的模型留給需要複雜推理的情況。 |
| **快取** | 快取工具結果和常見查詢，以避免重複的 API 呼叫。 |
| **Token 預算** | 設定 `max_tokens` 限制以防止回應意外過長。 |
| **批次處理** | 在可能的情況下，將多個使用者查詢合併成單一的 API 呼叫。 |

在實務上，採用分層方法效果良好：把簡單的請求導向快速且廉價的模型，只有複雜的查詢才升級到更強大的模型。


## 摘要

在本課程中，你學會了如何：

1. **加入可觀察性**於代理互動，透過計時與日誌記錄，為追蹤與監控奠定基礎。
2. **自動評估代理回應**，使用評估代理自動對完整性、準確性與有用性進行評分。
3. **管理成本**，透過提示優化、模型選擇、快取與 token 預算。

這些生產模式有助於確保你的 AI 代理在大規模部署時可靠、可衡量且具成本效益。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件已使用 AI 翻譯服務（Co-op Translator：https://github.com/Azure/co-op-translator）進行翻譯。儘管我們致力於確保準確性，請注意自動翻譯可能包含錯誤或不準確之處。原始語言版本的文件應被視為具權威性的來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤譯負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
